<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.1-spm-and-thermal/Ex10.1_05_report.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.1 · Notebook 05 — assemble the report

**Paired with L10.1 · Battery models**

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.1-spm-and-thermal/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
print("PyBaMM available:", pb.PYBAMM_OK)
if not pb.PYBAMM_OK:
    print("  install it with:   !pip install -q pybamm")

## 1 · What the other notebooks produced

In [ ]:
import pickle
runs = []
for f in ("ex101_spm.pkl", "ex101_runs.pkl"):
    p = os.path.join("Ex10.1_outputs", f)
    if os.path.exists(p):
        with open(p, "rb") as fh:
            d = pickle.load(fh)
        runs.extend(d if isinstance(d, list) else [d])
        print(f"  loaded  {f}")
    else:
        print(f"  MISSING {f} -- run that notebook first")
print(f"{len(runs)} runs loaded")

## 2 · Your personal seed

The notebooks fix the seed to 88 so that a result can be repeated on any
machine. The report asks for numbers from **your** seed instead, so a report
cannot be copied between groups without the numbers giving it away.

In [ ]:
STUDENT_NUMBER = "20241234"        # <- your AAU study number

SEED = personal_seed(STUDENT_NUMBER)
print("study number :", STUDENT_NUMBER)
print("your seed    :", SEED)

# Your own draw through the (r, t) slab, and what the analytic solution does
# on it. Quote these in the report.
your_pts = pb.particle_points(3000, t_end=1.0, seed=SEED)
c_you = pb.analytic_sphere(your_pts[:, 0], your_pts[:, 1])

print()
print(f"  your mean concentration : {c_you.mean():.5f}")
print(f"  your peak concentration : {c_you.max():.5f}")

## 3 · Assemble

In [ ]:
os.makedirs("Ex10.1_outputs", exist_ok=True)
path = pb.make_report(runs,
                      filename=os.path.join("Ex10.1_outputs", "Ex10.1_report.md"),
                      author="YOUR NAME",
                      notes=f"study number {STUDENT_NUMBER}, seed {SEED}")
# from google.colab import files; files.download(path)
print(open(path).read()[:1600])

## Submitting

Answer the five questions in the generated report. Question 5 asks you to make
the case for building a PINN when PyBaMM solves the same problem faster — see
L10.1 slide 21, and answer it in your own words rather than repeating the slide.

## Extensions

- Make `Ds` trainable and recover it from a voltage curve (slide 18).
- Add a relaxation period after the current step and show that it identifies
  `Ds` far better than steady discharge does.
- Replace the lumped thermal model with a trained r–z field and compare.
- Run at −10 °C ambient and explain why the cell behaves so differently.